# Module 2 — Using Claude Code for String + File Processing

**SBML Lab Intern Training**

---

This module shows how Claude Code can generate file-processing and data-manipulation scripts from natural language descriptions.

The core skill is simple:

> **Describe what you want in plain English → get a script → refine it with follow-ups.**

By the end of this module you will have practiced this loop on GFF file parsing and pandas workflows — two tasks you will encounter constantly in the lab.


## Background: Genome Annotations and the GFF Format

### What is a genome annotation?

A genome is just a long string of DNA — billions of base pairs with no obvious structure by itself. A **genome annotation** is a map that labels where things are: this region is a gene, this region encodes a protein. Without an annotation, you have sequence but no biology.

The lab's annotation for *E. coli* K-12 MG1655 marks every gene in the genome — about 4,600 features total (all rows are `gene` features; this file has no separate `CDS` rows). This file is the reference you'll work with throughout the training.

### The GFF format

GFF (General Feature Format) is the standard file format for genome annotations. Every row describes one genomic feature (a gene, CDS, etc.) using 9 tab-separated columns:

The columns below describe the GFF format in general; the **Example** column shows the actual values found in this lab annotation file:

| Column | Name | Example (this file) |
|--------|------|---------|
| 1 | Chromosome / sequence name | `NC_000913` |
| 2 | Source | `Ecocyc_14.1` |
| 3 | Feature type | `gene` |
| 4 | Start position | `337` |
| 5 | End position | `2799` |
| 6 | Score | `.` (unused here) |
| 7 | Strand | `+` or `-` |
| 8 | Phase | `.` |
| 9 | Attributes | `locus_tag=b0002;gene=thrA;product=...;color=00FF00` |

> **Note the attribute keys:** this file uses `locus_tag=` and `gene=` (not `ID=`/`Name=`). Parse for those keys when extracting gene names.
>
> **Sequence-name convention:** this annotation uses `NC_000913` (no version suffix). The reference genome you download in Module 3 is `NC_000913.3`. The suffix differs — this matters when you join files by chromosome (Module 4), where `NC_000913` and `NC_000913.3` will not match unless you reconcile them.

**Coordinates are 1-based** — position 1 is the first nucleotide of the chromosome. This matters when writing Python code, since Python string indexing is 0-based. An off-by-one error here silently shifts every position by one.

### Why does this lab use GFF?

GFF is the input format for **MetaScope**, the lab's genome browser for visualizing ChIP-exo data. The alignment pipeline (Module 3) produces a GFF file as its final output so reads can be loaded into MetaScope alongside the gene annotation.

---

## GFF Format Recap

GFF is the standard annotation format used in this lab — tab-delimited, 9 columns, no header row, 1-based coordinates. Use `lab-context.md` as a reference if needed.

**Data for this module:** the lab *E. coli* K-12 MG1655 annotation GFF is pre-provided at:
```
data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff
```
This is the same file used in Modules 4 and 5 (the mini-project).


## Natural Language → Script

The pattern you will use throughout this module:

1. **Write a plain-English description** of what you want the script to do. Be specific: mention the file format, the columns involved, and the expected output.
2. **Claude Code generates a first draft.** Paste it into a code cell and run it.
3. **Follow up with refinements.** Fix edge cases, add output formatting, handle errors, change the logic.

> **Important:** your first prompt is a draft, not a final answer. Follow-up prompts are where the real work happens. Most production scripts in the lab went through 3–10 iterations before they were committed.

### Tips for better first prompts

- Name the file format explicitly: *"a GFF file"*, not *"a text file"*.
- State the output: *"print the result to stdout"* or *"save to a new TSV file"*.
- Mention what to do with edge cases: *"skip comment lines"*, *"ignore rows where score is `.`"*.

---

### Why would you filter a genome annotation?

Real analyses rarely use the entire annotation. Common filtering scenarios in this lab:

- **By strand:** ChIP-exo reads are strand-specific — a binding site on the `+` strand regulates genes on the `+` strand. You often want to look at only one strand at a time.
- **By feature type:** You might want only `gene` entries, not `CDS`, depending on the question.
- **By coordinate range:** If you're zooming in on a genomic region (say, the first 500 kb), you only want features within those coordinates.

### What does "strand" mean?

The *E. coli* chromosome is double-stranded DNA. Genes can be encoded on either strand:

- **`+` strand (forward):** The gene is read left-to-right as written in the reference FASTA. Start < End.
- **`-` strand (reverse):** The gene is read right-to-left. In GFF, start is still less than end (by convention), but the strand column is `-`.

This matters biologically because a TF binding site upstream of a `+` strand gene is in a different genomic direction than one upstream of a `-` strand gene.

---

### Exercise 1 — Generate and refine a GFF filtering script

1. Describe a GFF filtering task in plain English to Claude Code. Use the lab annotation GFF at `data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff`. For example: filter by feature type, by strand, or by coordinate range — your choice.
2. Generate the script.
3. Improve it with **at least two follow-up prompts**.

Write the **final version** of your script in the code cell below. Add a comment at the top of the script that briefly describes what each follow-up prompt changed.

Example comment format:
```python
# Follow-up 1: added argparse so the input path is a CLI argument
# Follow-up 2: changed output to TSV instead of printing to stdout
```

> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [ ]:
from pathlib import Path

GAP_THRESHOLD = 50  # 보통 50 bp 이내는 같은 operon으로 간주. operon끼리 모여있도록 filtering.  

def find_repo_file(relative_path, start=None): # claude가 파일을 찾는 함수. relative_path를 기준으로 start 경로에서 상위 파일로 올라가며 파일을 찾는다.
    start = Path(start or Path.cwd()).resolve() # current working directory를 기준으로 start 경로를 설정. 
    for directory in [start, *start.parents]: 
        candidate = directory / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {relative_path!r} at or above {start}")

input_path = find_repo_file("data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff") # relative_path 변수로 들어가는 값. 
output_path = input_path.parent / "ec_annotation_operon_grouped.gff"  # 새 파일 생성. 원래 파일은 그대로. 

genes = [] # list 생성해 gff 파일의 한 줄씩 넣는다. 리스트를 만드는 이유는 각 유전자의 4번째 컬럼(시작 좌표)을 이용해 filtering 하므로.
with open(input_path) as f: # 파일 열기(자동으로 닫기 위해 with 사용).
    for line in f:
        if not line.strip():
            continue
        genes.append(line.rstrip("\n").split("\t"))

# 생성된 gene 리스트의 4번째 컬럼 (시작 좌표)을 starts 리스트에 저장. 오름차순으로 정렬되었는지 확인.
starts = [int(g[3]) for g in genes]
assert starts == sorted(starts), "file is not sorted by start position"

groups = [] # list 생성해 operon 그룹을 저장.
current = [genes[0]] # 첫 번째 유전자를 current 리스트에 넣는다.
for prev, gene in zip(genes, genes[1:]): # prev와 gene을 zip으로 묶어 비교. 
    same_strand = gene[6] == prev[6] # 같은 strand인지 확인. (6번째 컬럼이 strand).
    gap = int(gene[3]) - int(prev[4]) # gap 계산(현재 유전자 시작 좌표 - 이전 유전자 끝 좌표).
    if same_strand and gap <= GAP_THRESHOLD: 
        current.append(gene) # 같은 strand이고 gap이 threshold 이하면 current 리스트에 gene 추가.
    else:
        groups.append(current)
        current = [gene]  # 그렇지 않으면 current 리스트를 groups에 추가, current를 새 gene으로 초기화.
groups.append(current) # 남은 current 리스트를 groups에 추가.

# follow-up 1: operon_id와 operon_size를 attr에 추가하도록 한다.
with open(output_path, "w") as f: # 새 파일을 쓰기 모드로 열기.
    for op_idx, group in enumerate(groups, start=1): # enumerate 함수로 groups를 돌며 indexing. 
        for gene in group:
            attr = gene[8] + f";operon_id=op{op_idx:05d};operon_size={len(group)}"
            f.write("\t".join(gene[:8] + [attr]) + "\n") # attr에 operon_id와 operon_size를 추가.

# follow-up 2: input, output 과 total genes, total candidate operon groups 를 출력하도록 한다. 
print(f"input:  {input_path}")
print(f"output: {output_path}")
print(f"total genes: {len(genes)}")
print(f"total candidate operon groups: {len(groups)}")


input:  /workspaces/intern-claude-training/data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff
output: /workspaces/intern-claude-training/data/reference/ec_annotation_operon_grouped.gff
total genes: 4595
total candidate operon groups: 2907


## Loading GFF Files with pandas

Loading a GFF file with pandas requires a few non-obvious options. Use Claude Code to figure out the correct `read_csv` call for `data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff`.

> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [6]:
from pathlib import Path # 파일 경로 지정을 위해 pathlib 모듈을 import.
import pandas as pd

def find_repo_file(relative_path, start=None):
    start = Path(start or Path.cwd()).resolve()
    for directory in [start, *start.parents]:
        candidate = directory / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {relative_path!r} at or above {start}")

input_path = find_repo_file("data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff") # relative_path 변수로 들어가는 값.

# GFF 파일을 읽기 위해서는 단순히 pandas의 read_csv 함수를 사용하면 안 된다. GFF 파일은 탭으로 구분되며, 헤더가 없으므로 header=None으로 설정하고, 컬럼 이름을 지정해줘야 한다.
df = pd.read_csv(input_path,
                 sep="\t",
                 header=None,
                 names=["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"])

df.head() # 데이터프레임의 상위 5개 행을 출력.

,seqid,source,type,start,end,score,strand,phase,attributes
0,NC_000913,Ecocyc_14.1,gene,190,255,.,+,.,locus_tag=b0001;gene=thrL;feature=gene;product...
1,NC_000913,Ecocyc_14.1,gene,337,2799,.,+,.,locus_tag=b0002;gene=thrA;feature=gene;product...
2,NC_000913,Ecocyc_14.1,gene,2801,3733,.,+,.,locus_tag=b0003;gene=thrB;feature=gene;product...
3,NC_000913,Ecocyc_14.1,gene,3734,5020,.,+,.,locus_tag=b0004;gene=thrC;feature=gene;product...
4,NC_000913,Ecocyc_14.1,gene,5234,5530,.,+,.,locus_tag=b0005;gene=yaaX;feature=gene;product...


### Exercise 2 — Debugging: Loading and Indexing

The script below has **two bugs**. Use Claude Code to diagnose and fix them.

Workflow:
1. Read the script and try to spot the bugs yourself first.
2. Paste the script into Claude Code and ask it to find the bugs.
3. Use `/debug` in the Claude Code terminal if you get stuck.
4. Write the corrected version in the empty code cell below the broken one.

> **Hint:** there is one bug in how the file is loaded, and one bug in how a row is accessed.

> **Explain it:** after fixing it, write 1–2 sentences on *what each bug was and why your fix works*. Understanding the bug matters more than the patch.

In [9]:
import pandas as pd

# Load GFF file
df = pd.read_csv('data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff', sep='\t')

# Filter for forward strand genes
forward = df[df['strand'] == '+']

# Get the 5th feature
fifth = forward.iat[5]

print(f"5th forward feature starts at position {fifth['start']}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff'

In [12]:
# 1. FileNotFoundError: 기존 코드는 상대 경로를 사용했다. Path.cwd()를 사용하면 현재 작업 디렉토리를 기준으로 파일을 찾아 에러가 난다. 따라서 find_repo_file() 을 통해 파일의 절대 경로를 찾는다.
from pathlib import Path
import pandas as pd

def find_repo_file(relative_path, start=None):
    start = Path(start or Path.cwd()).resolve()
    for directory in [start, *start.parents]:
        candidate = directory / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {relative_path!r} at or above {start}")

input_path = find_repo_file("data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff")

# 2. pandas에서 .ist[]는 df의 특정 행과 열이 교차하는 단 하나의 원소값을 정수 위치 기반으로 가져올 때 쓰는 method이다. 따라서 행 추출을 위해서는 .iloc[]를 사용해야 한다. 또한 pandas는 숫자를 0부터 세는 0-based indexing이므로 5번째 feature는 index 4가 된다. 
df = pd.read_csv(input_path, sep='\t',
                  header=None,
                  names=["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"])

forward = df[df['strand'] == '+']

fifth = forward.iloc[5]

print(f"5th forward feature starts at position {fifth['start']}")

5th forward feature starts at position 8238


## `iterrows()` — When and Why

`iterrows()` iterates over a DataFrame row by row. Use Claude Code to understand when it is appropriate and when to avoid it.

### Exercise 3a — When to use it (concept)

After getting Claude Code's explanation, write a **2-sentence answer** in the markdown cell below:
- Sentence 1: when you **should** use `iterrows()`.
- Sentence 2: when you **should avoid** it and what to use instead.

### Exercise 3b — Use it on real data (hands-on)

Now actually iterate. Using the annotation DataFrame you loaded above, **find the gene whose transcription start site (TSS) is closest to position 1,000,000** on the genome.

- For a `+`-strand gene the TSS is its `start`; for a `-`-strand gene the TSS is its `end`.
- Loop over rows with `df.iterrows()`, compute each gene's distance to 1,000,000, and report the closest gene's name (the `gene=` value in the attribute column) and its distance.

Write the working script in the code cell below. When it runs, ask Claude Code how you would do the *same* computation **without** `iterrows()` (vectorized) — this is the practical version of the concept from Exercise 2a.


> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [15]:
# Exercise 3b — find the gene whose TSS is closest to position 1,000,000, using df.iterrows()
# Your script here.

#3a의 답변: 
#sentence 1: you should use iterrows() when the per-row logic is inherently sequential, where each row depends on the previous one, or when performing non-vectorizable actions like row-by-row debugging or calling external functions. 
#sentence 2: you should avoid it for any column-wide operations because creating a new Series for every row is drastically slow, and you should use pandas vectorized column math or .apply() instead. 

#3b의 답변: 
import re

target_position = 1_000_000
closest_gene_name = None
min_distance = float('inf')

for index, row in df.iterrows():
    
    if pd.isna(row['strand']):
        continue
        
    tss_position = row['start'] if row['strand'] == '+' else row['end']
    distance = abs(tss_position - target_position)
    
    if distance < min_distance:
        min_distance = distance
        
        attr_str = str(row['attributes'])
        match = re.search(r'gene=([^;]+)', attr_str)
        if match:
            closest_gene_name = match.group(1)
        else:
            closest_gene_name = attr_str 

print(f"Closest Gene Name: {closest_gene_name}")
print(f"Distance: {min_distance}")

Closest Gene Name: ycbT
Distance: 1030


> **Answer**
># Vectorized TSS distance calculation (no iterrows())

tss = df['start'].where(df['strand'] == '+', df['end'])

distance = (tss - 1_000_000).abs()

closest_idx = distance.idxmin()
closest_gene = df.loc[closest_idx]

gene_name = closest_gene['attributes'].split('gene=')[1].split(';')[0]
print(f"Closest gene: {gene_name}, distance: {distance.loc[closest_idx]}")
> *Your 2-sentence answer here.*
**작동 원리 및 선택 이유:**
벡터화 코드는 전체 행의 strand를 확인해 TSS 열을 동시에 생성하고 1,000,000을 빼 절대 거리를 한 번의 행렬 연산으로 계산한다. 그 후 `.idxmin()`을 통해 가장 가까운 행의 인덱스를 뱉는다. 
itterrows() 연산은 for 루프로 데이터가 4600개가 있으면 4600번을 실행하여 속도가 느리다. 또한 iterrows()의 row 변수는 원본 df가 아닌 복사본으로 수정해도 원본 데이터가 변하지 않는다. 따라서 vectorized 연산을 하는 것이 빠른 속도로 연산을 수행할 때 더욱 유리하다. 




## End of Session

Before closing, run `/log` in the Claude Code terminal to save a summary of your session.

This creates a record of the prompts you used and the scripts you generated — useful for your own reference and for lab documentation.

## Git — Commit Your Work

Every session ends with a commit. Run the commands below in your terminal (not in this notebook).

Write your own commit message following the format: `feat(module2): <what you did>`.  
If you're unsure what to write, ask Claude Code to suggest one based on what you worked on.

In [ ]:
# Run these in the terminal (not here):
# git add notebooks/
# git commit -m "..."   # write your own commit message; use Claude Code to help if needed
# git push
